In [ ]:
# Data sources:
# Brussels Mobility website: in 2022, there were 52,204 company cars registered in the Brussels-Capital Region
# https://be.brussels/fr/transport-mobilite/enjeux-de-la-mobilite/observatoire-thematique/automobile/equipements-automobiles-des-menages#5

# Brussels Mobility report: company car by age
# Enquête relative au besoin des bruxellois de se déplacer en voiture
# https://admin.be.brussels/sites/default/files/2025-02/enquete_auto_FR.pdf

# DDT 2024: company car availability by economic sector
# https://mobilit.belgium.be/fr/publications/enquete-federale-sur-les-deplacements-domicile-travail-2024-2025-0

# IBSA Focus 53: percentage of households with company cars by municipality (2019 data)
# https://ibsa.brussels/sites/default/files/publication/documents/Focus-53_FR_0.pdf

In [ ]:
import pandas as pd

In [ ]:
# Load data
pop = pd.read_csv('../4_car_ownership/output/workers_with_cars.csv')

In [ ]:
# Define the company car ownership rates/probabilities by:
# 1. Municipality
# 2. Economic sector
# 3. Age group

avg_muni_weight = 0.077
municipality_weights = {
    '21001': 0.039 / avg_muni_weight,   # Anderlecht
    '21002': 0.149 / avg_muni_weight,   # Auderghem
    '21003': 0.083 / avg_muni_weight,   # Berchem-Sainte-Agathe
    '21004': 0.059 / avg_muni_weight,   # Ville de Bruxelleslles
    '21005': 0.087 / avg_muni_weight,   # Etterbeek
    '21006': 0.076 / avg_muni_weight,   # Evere
    '21007': 0.072 / avg_muni_weight,   # Forest
    '21008': 0.062 / avg_muni_weight,   # Ganshoren
    '21009': 0.085 / avg_muni_weight,   # Ixelles
    '21010': 0.067 / avg_muni_weight,   # Jette
    '21011': 0.050 / avg_muni_weight,   # Koekelberg
    '21012': 0.040 / avg_muni_weight,   # Molenbeek-Saint-Jean
    '21013': 0.052 / avg_muni_weight,   # Saint-Gilles
    '21014': 0.026 / avg_muni_weight,   # Saint-Josse-ten-Noode
    '21015': 0.066 / avg_muni_weight,   # Schaerbeek
    '21016': 0.130 / avg_muni_weight,   # Uccle
    '21017': 0.125 / avg_muni_weight,   # Watermael-Boitsfort
    '21018': 0.127 / avg_muni_weight,   # Woluwe-Saint-Lambert
    '21019': 0.159 / avg_muni_weight,   # Woluwe-Saint-Pierre
}

age_bracket_weight = 0.16
age_group_weights = {
    (15, 24): 0.11 / age_bracket_weight,
    (25, 34): 0.20 / age_bracket_weight,
    (35, 44): 0.21 / age_bracket_weight,
    (45, 54): 0.25 / age_bracket_weight,
    (55, 64): 0.21 / age_bracket_weight,
    (65, 199): 0.04 / age_bracket_weight,
}

avg_sector_weight = 0.53
sector_weights = {
    'J':    0.60 / avg_sector_weight,
    'K':    0.49 / avg_sector_weight,
    'M_N':  0.31 / avg_sector_weight,   # M = 0.44, N = 0.18, so we take average for M-N = 0.31
    'F':    0.43 / avg_sector_weight,
    'G-I':  0.14 / avg_sector_weight,   # G = 0.25, H = 0.08, I = 0.09, so we take average for G-I = 0.14
    'B-E':  0.20 / avg_sector_weight,   # B = 0.17, C = 0.19, D = 0.32, E = 0.11, so we take average for B-E = 0.20
    'A':    0.16 / avg_sector_weight,
    'R-U':  0.08 / avg_sector_weight,   # R = 0.14, S = 0.18, T = 0.0, U = 0.0, so we take average for R-U = 0.08
    'L':    0.13 / avg_sector_weight,
    'O-Q':  0.01 / avg_sector_weight,   # O = 0.01, P = 0.01, Q = 0.02, so we take average for O-Q = 0.01
    'UNK':  1.0,   # Unknown sector, we assume the average company car ownership
}

In [ ]:
# Define necessary functions to map the agents to the weight groups
def extract_municipality_code(sector_id):
    """Extract 5-digit NIS municipality code from sector ID string."""
    return str(sector_id)[:5]

def parse_age_bracket(age_str):
    """Extract lower bound from age bracket string like '25-29', '100+'."""
    if pd.isna(age_str):
        return 0
    age_str = str(age_str).strip()
    if '+' in age_str:
        return int(age_str.replace('+', ''))
    return int(age_str.split('-')[0])

def get_age_adjustment(age_lower):
    for (lo, hi), adj in age_group_weights.items():
        if lo <= age_lower <= hi:
            return adj
    return 0.0

In [ ]:
# Filter to only keep agents who were already assigned a car in the previous step
eligible_agents = pop[pop['has_car'] == 1].copy()
eligible_agents['municipality'] = eligible_agents['assigned_sector'].apply(extract_municipality_code)
eligible_agents['age_bracket'] = eligible_agents['age'].apply(parse_age_bracket)

In [ ]:
# Calculate composite score
eligible_agents['w_sector'] = eligible_agents['industry'].map(sector_weights).fillna(0.01)
eligible_agents['w_age'] = eligible_agents['age_bracket'].apply(get_age_adjustment).fillna(0.01)
eligible_agents['w_muni'] = eligible_agents['municipality'].map(municipality_weights).fillna(0.01)

eligible_agents['P_cc'] = (eligible_agents['w_sector'] * eligible_agents['w_age'] * eligible_agents['w_muni']) + 1e-6

In [ ]:
# Run the weighted random sampling to assign company cars
TARGET_COMPANY_CARS = 52204

# Sample 52,204 agents without replacement, using P_cc as the probability weights
company_car_winners = eligible_agents.sample(
    n=TARGET_COMPANY_CARS, 
    replace=False, 
    weights='P_cc',
    random_state=42
)

In [ ]:
# Initialise the company car field and assign company cars to the winners
pop['has_company_car'] = False

pop.loc[company_car_winners.index, 'has_company_car'] = True

In [ ]:
# Save results
pop.to_csv('output/workers_with_company_cars.csv', index=False)